# Fronteira de Pareto Separada — Para TODOS os Indicadores de Classificação

Generaliza o notebook 6 (`6_FronteiraParetoSeparada.ipynb`), que fazia a comparação
Proposta vs. (Literatura + Descritivas) usando só a Acurácia Balanceada. **Este notebook
repete a mesma análise para cada indicador de desempenho disponível** (Sensibilidade,
Especificidade, Acurácia, AUC, Precisão, Recall e F1-Score, nas versões Macro e
Weighted) -- não só a Acurácia Balanceada.

**Objetivo**: analisar **todos** os indicadores de uma vez, reportando igualmente onde a proposta vence, perde ou empata em cada um, para caracterizar **em quais contextos (paradigma × indicador) as métricas da proposta têm vantagem e em quais não têm**. Nenhum indicador é descartado.

**O que NÃO muda em relação ao notebook 6**: os dois cotovelos por ranking (Kneedle +
Estatístico) continuam sendo escolhidos com base na **Acurácia Balanceada** -- é essa a
metodologia de seleção de modelo já estabelecida na tese (o "modelo escolhido" em cada
ranking). O que muda é que, nesses MESMOS pontos já selecionados, agora extraímos e
comparamos **todos** os outros indicadores também, não só a Acurácia Balanceada que
serviu para escolher o ponto.

**Saída em pastas por indicador**: cada indicador tem sua própria subpasta dentro de
`Resultados_DuplaFronteira_PorIndicador/` (ex.: `Sensibilidade/`, `AUC/`,
`Precisao_Macro/`), com os mesmos tipos de arquivo que o notebook 6 já gerava (gráficos,
tabela de pontos na fronteira, tabela de todos os pontos, comparação Proposta vs. Resto).
Ao final, um `Resumo_Todos_Indicadores.xlsx` na raiz consolida o resultado de **todos**
os indicadores e paradigmas numa única tabela, para ver de uma vez só onde a
proposta tem vantagem e onde não tem.

**Como usar**: copie este notebook para a mesma pasta onde estão os
`Modelos<Paradigma>_CV_Consolidado.xlsx` (saída do notebook 4 já consolidado) e rode
todas as células, em ordem.

### 1. Configurações, leitura dos arquivos consolidados e cálculo dos cotovelos

In [1]:
import os
import re
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from datetime import datetime

FOLDER_PATH = os.getcwd()
OUT_DIR_ROOT = os.path.join(FOLDER_PATH, "Resultados_DuplaFronteira_PorIndicador")
os.makedirs(OUT_DIR_ROOT, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

ORDEM_PARADIGMAS = ["DT", "RF", "GB", "SVM", "MLP", "GNB", "LogReg_Lasso"]

print(f"Diretório base: {FOLDER_PATH}")
print(f"Salvando resultados em: {OUT_DIR_ROOT}")


def extrair_paradigma(filename):
    '''Modelos<Paradigma>_CV_Consolidado.xlsx -> <Paradigma> (ex.: LogReg_Lasso) --
    não-guloso até o primeiro _CV_, para funcionar mesmo com underscore no nome do
    paradigma.'''
    m = re.match(r"^Modelos(.+?)_CV_", filename)
    return m.group(1) if m else "Desconhecido"


REGEX_CEQ = re.compile(r"^(Cp|Cpk|Pp|Ppk|p|np|c|u)_(Original|Normalizado)$")
PREFIXOS_DESCRITIVAS = ("AmplitudeSigma", "IQRSigma", "Assimetria", "Curtose")


def classificar_grupo(nome_ranking):
    '''Classifica um ranking em "Proposta", "Descritiva" ou "Literatura".
    Retorna None para as versões "_BaseCompleta" das descritivas -- excluídas da
    análise por não serem pareadas com a mesma população (classe de anomalia) que
    todo o resto usa. Espera nome_ranking já sem o prefixo "Ranking_" (ex.:
    "Cp_Original", não "Ranking_Cp_Original").'''
    if "BaseCompleta" in nome_ranking:
        return None
    if REGEX_CEQ.match(nome_ranking):
        return "Proposta"
    if nome_ranking.startswith(PREFIXOS_DESCRITIVAS):
        return "Descritiva"
    return "Literatura"


# --- Cotovelos escolhidos SEMPRE pela Acurácia Balanceada (metodologia de seleção já
# estabelecida na tese) -- os outros indicadores só são LIDOS nesses mesmos pontos,
# nunca usados para re-escolher um cotovelo diferente. ---
COL_X = "Nº de Atributos"
COL_Y_SELECAO = "Acurácia Balanceada Média"
COL_STD_SELECAO = "Acurácia Balanceada Desvio"


def get_kneedle_idx(df_sorted, col_x=COL_X, col_y=COL_Y_SELECAO):
    '''Ponto de maior distância perpendicular à reta que liga o primeiro e o
    último ponto da curva (normalizada em [0,1] nos dois eixos). Produto
    vetorial 2D calculado manualmente (ax*by - ay*bx), não via np.cross, que a
    partir do NumPy 2.4 passou a exigir vetores 3D.'''
    x = df_sorted[col_x].values.astype(float)
    y = df_sorted[col_y].values.astype(float)
    if x.max() == x.min() or y.max() == y.min() or len(x) < 3:
        return df_sorted.index[0]
    x_norm = (x - x.min()) / (x.max() - x.min())
    y_norm = (y - y.min()) / (y.max() - y.min())
    coords = np.vstack((x_norm, y_norm)).T
    vec_line = coords[-1] - coords[0]
    vec_points = coords - coords[0]
    norm = np.linalg.norm(vec_line)
    if norm == 0:
        return df_sorted.index[0]
    distances = (vec_line[0] * vec_points[:, 1] - vec_line[1] * vec_points[:, 0]) / norm
    return df_sorted.index[np.argmax(np.abs(distances))]


def get_estatistico_idx(df_sorted, col_y=COL_Y_SELECAO, col_std=COL_STD_SELECAO):
    '''Regra de 1 desvio-padrão: entre os pontos cuja acurácia fica a até 1
    desvio-padrão da melhor acurácia da curva, escolhe o de menor N (modelo
    mais simples estatisticamente indistinguível do melhor).'''
    idx_melhor = df_sorted[col_y].idxmax()
    melhor_val = df_sorted.loc[idx_melhor, col_y]
    desvio = df_sorted.loc[idx_melhor, col_std] if col_std in df_sorted.columns and pd.notna(df_sorted.loc[idx_melhor, col_std]) else 0.0
    limiar = melhor_val - desvio
    candidatos = df_sorted[df_sorted[col_y] >= limiar]
    return candidatos[COL_X].idxmin()


# --- Indicadores analisados: (rótulo da coluna "<rótulo> Média"/"<rótulo> Desvio"/
# "<rótulo>_Por_Fold" no Excel do notebook 4, slug usado como nome de pasta) ---
INDICADORES = [
    ("Acurácia Balanceada",      "Acuracia_Balanceada"),
    ("Sensibilidade",            "Sensibilidade"),
    ("Especificidade",           "Especificidade"),
    ("Acurácia",                 "Acuracia"),
    ("AUC",                      "AUC"),
    ("Precisão (Macro Avg)",     "Precisao_Macro"),
    ("Recall (Macro Avg)",       "Recall_Macro"),
    ("F1-Score (Macro Avg)",     "F1_Macro"),
    ("Precisão (Weighted Avg)",  "Precisao_Weighted"),
    ("Recall (Weighted Avg)",    "Recall_Weighted"),
    ("F1-Score (Weighted Avg)",  "F1_Weighted"),
]

arquivos_consolidados = sorted([
    f for f in os.listdir(FOLDER_PATH)
    if f.startswith("Modelos") and f.endswith("_CV_Consolidado.xlsx")
])
print(f"\n{len(arquivos_consolidados)} arquivo(s) 'Modelos<Paradigma>_CV_Consolidado.xlsx' encontrado(s).")

# df_all_ind: formato LONGO -- uma linha por (Paradigma, Ranking, Grupo, Tipo, Indicador),
# com o valor do indicador naquele cotovelo (e os 5 valores de fold, se disponíveis).
registros = []
for arquivo in arquivos_consolidados:
    paradigma = extrair_paradigma(arquivo)

    try:
        todas_as_abas = pd.read_excel(os.path.join(FOLDER_PATH, arquivo), sheet_name=None)
    except Exception as e:
        print(f"Erro ao ler {arquivo}: {e}")
        continue

    for nome_aba, df in todas_as_abas.items():
        nome_ranking = nome_aba[len("Ranking_"):] if nome_aba.startswith("Ranking_") else nome_aba

        grupo = classificar_grupo(nome_ranking)
        if grupo is None:
            continue  # _BaseCompleta -- fora da análise

        if COL_X not in df.columns or COL_Y_SELECAO not in df.columns:
            continue

        df_sorted = df.sort_values(COL_X, ascending=True).reset_index(drop=True)
        if len(df_sorted) < 2:
            continue

        idx_kneedle = get_kneedle_idx(df_sorted)
        idx_estatistico = get_estatistico_idx(df_sorted)

        for tipo, idx in [("Kneedle", idx_kneedle), ("Estatístico", idx_estatistico)]:
            linha = df_sorted.loc[idx]
            n_atributos = linha[COL_X]

            for label, slug in INDICADORES:
                col_media = f"{label} Média"
                col_fold = f"{label}_Por_Fold"
                if col_media not in df_sorted.columns:
                    continue
                registros.append({
                    "Paradigma": paradigma, "Ranking": nome_ranking, "Grupo": grupo, "Tipo": tipo,
                    "Nº de Atributos": n_atributos,
                    "Indicador": label, "Indicador_Slug": slug,
                    "Valor": linha[col_media],
                    "Valores_Por_Fold": linha.get(col_fold, None),
                })

df_all_ind = pd.DataFrame(registros).drop_duplicates(
    subset=["Paradigma", "Ranking", "Tipo", "Indicador"]
) if registros else pd.DataFrame()

print(f"{len(df_all_ind)} registro(s) (Paradigma x Ranking x Tipo x Indicador) carregados.")
if not df_all_ind.empty:
    print(f"   Indicadores disponíveis: {sorted(df_all_ind['Indicador'].unique())}")
    print(f"   Paradigmas disponíveis:  {sorted(df_all_ind['Paradigma'].unique())}")


Diretório base: c:\Users\USUARIO\Desktop\doutorado\Códigos para o Artigo
Salvando resultados em: c:\Users\USUARIO\Desktop\doutorado\Códigos para o Artigo\Resultados_DuplaFronteira_PorIndicador

7 arquivo(s) 'Modelos<Paradigma>_CV_Consolidado.xlsx' encontrado(s).
✅ 6160 registro(s) (Paradigma x Ranking x Tipo x Indicador) carregados.
   Indicadores disponíveis: ['AUC', 'Acurácia', 'Acurácia Balanceada', 'Especificidade', 'F1-Score (Macro Avg)', 'F1-Score (Weighted Avg)', 'Precisão (Macro Avg)', 'Precisão (Weighted Avg)', 'Recall (Macro Avg)', 'Recall (Weighted Avg)', 'Sensibilidade']
   Paradigmas disponíveis:  ['DT', 'GB', 'GNB', 'LogReg_Lasso', 'MLP', 'RF', 'SVM']


### 2. Função da fronteira de Pareto (genérica -- qualquer indicador como eixo Y)

In [2]:
def get_pareto_frontier(df, col_x="Nº de Atributos", col_y="Valor"):
    '''Fronteira de Pareto para minimizar N e maximizar o indicador em col_y: ordena
    por N crescente e mantém só os pontos cujo valor supera tudo que veio antes.'''
    df_sorted = df.sort_values(col_x, ascending=True).reset_index(drop=True)
    pareto_idx = []
    melhor_valor = -np.inf
    for i, row in df_sorted.iterrows():
        if row[col_y] > melhor_valor:
            pareto_idx.append(i)
            melhor_valor = row[col_y]
    return df_sorted.loc[pareto_idx].reset_index(drop=True)


### 3. Gráficos e tabelas, para CADA indicador e CADA paradigma

Mesma lógica do notebook 6 (duas fronteiras -- Proposta sozinha vs. Literatura+Descritiva
juntas -- com jitter visual e diagnóstico de sobreposição), repetida para cada indicador
da lista `INDICADORES` acima. Cada indicador grava seus arquivos numa subpasta própria,
nomeada com o slug do indicador.

In [3]:
COR_GRUPO = {"Proposta": "#1f77b4", "Literatura": "#2ca02c", "Descritiva": "#ff7f0e"}
MARCADOR_TIPO = {"Kneedle": "o", "Estatístico": "s"}


def plot_fronteira(df_grupo, titulo, caminho_arquivo, cores_por_subgrupo=None):
    fig, ax = plt.subplots(figsize=(10, 7))

    if cores_por_subgrupo:
        grupos_cor = cores_por_subgrupo
    else:
        grupos_cor = {"Proposta": COR_GRUPO["Proposta"]}

    chave = df_grupo["Nº de Atributos"].astype(str) + "_" + df_grupo["Valor"].round(4).astype(str)
    contagem_sobreposicao = chave.value_counts()
    n_max_sobreposto = int(contagem_sobreposicao.max()) if len(contagem_sobreposicao) else 1

    rng_jitter = np.random.default_rng(42)
    for subgrupo, cor in grupos_cor.items():
        for tipo, marcador in MARCADOR_TIPO.items():
            sub = df_grupo[(df_grupo["Grupo"] == subgrupo) & (df_grupo["Tipo"] == tipo)]
            if sub.empty:
                continue
            jitter_x = rng_jitter.normal(0, 0.4, size=len(sub))
            jitter_y = rng_jitter.normal(0, 0.0015, size=len(sub))
            ax.scatter(sub["Nº de Atributos"] + jitter_x, sub["Valor"] + jitter_y,
                       c=cor, s=60, alpha=0.6, edgecolors="black", linewidths=0.5,
                       marker=marcador, label=f"{subgrupo} ({tipo})")

    if n_max_sobreposto > 1:
        ax.text(0.02, 0.02, f"até {n_max_sobreposto} pontos sobrepostos na mesma coordenada\n(jitter aplicado só para visualização)",
                transform=ax.transAxes, fontsize=8, color="gray", va="bottom")

    pareto = get_pareto_frontier(df_grupo)
    ax.plot(pareto["Nº de Atributos"], pareto["Valor"],
            "r--", linewidth=1.5, label="Fronteira de Pareto", zorder=1)

    ax.set_xlabel("Complexidade (Nº de Atributos)")
    ax.set_ylabel("Valor do Indicador")
    ax.set_title(titulo)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(caminho_arquivo, dpi=150)
    plt.close()
    return pareto


def tabela_pontos_na_fronteira(df_grupo_completo):
    '''Entre os cotovelos (Kneedle + Estatístico) do grupo, identifica quais caem
    exatamente sobre a fronteira de Pareto e monta a tabela de síntese.'''
    pareto = get_pareto_frontier(df_grupo_completo)
    pareto_tuplas = list(zip(pareto["Nº de Atributos"].values, pareto["Valor"].values))

    registros_tab = []
    for _, row in df_grupo_completo.iterrows():
        esta_na_fronteira = any(
            np.isclose(row["Nº de Atributos"], px, atol=1e-6) and
            np.isclose(row["Valor"], py, atol=1e-6)
            for px, py in pareto_tuplas
        )
        if esta_na_fronteira:
            registros_tab.append({
                "Grupo": row["Grupo"], "Método": f"{row['Ranking']} ({row['Tipo']})",
                "Dim": int(row["Nº de Atributos"]), "Valor": float(row["Valor"]),
            })
    df_r = pd.DataFrame(registros_tab).drop_duplicates()
    if df_r.empty:
        return df_r
    agrupado = df_r.groupby(["Grupo", "Dim", "Valor"])["Método"].apply(list).reset_index()

    def fmt_metodos(lista):
        if len(lista) == 1: return lista[0]
        if len(lista) <= 2: return ", ".join(lista)
        return f"{lista[0]}, {lista[1]} (+{len(lista)-2} empatados)"

    agrupado["Metodo_fmt"] = agrupado["Método"].apply(fmt_metodos)
    agrupado = agrupado.sort_values(["Dim", "Valor"]).reset_index(drop=True)
    return agrupado[["Grupo", "Metodo_fmt", "Dim", "Valor"]]


def exportar_todos_os_pontos(df_grupo, caminho):
    '''Exporta TODOS os pontos (não só os da fronteira), marcando sobreposição --
    útil para conferir manualmente se pontos estão coincidindo no gráfico.'''
    df_exp = df_grupo[["Grupo", "Ranking", "Tipo", "Nº de Atributos", "Valor"]].copy()
    df_exp = df_exp.rename(columns={"Nº de Atributos": "Dim"})
    df_exp["Valor_arred"] = df_exp["Valor"].round(4)
    contagem = df_exp.groupby(["Dim", "Valor_arred"])["Ranking"].transform("count")
    df_exp["Sobreposto"] = contagem > 1
    df_exp["Qtd_no_mesmo_ponto"] = contagem
    df_exp = df_exp.drop(columns="Valor_arred").sort_values(["Dim", "Valor"]).reset_index(drop=True)
    df_exp.to_excel(caminho, index=False)
    return df_exp


resumo_todos_indicadores = []  # consolida todas as comparações, de todos os indicadores

for label, slug in INDICADORES:
    df_ind = df_all_ind[df_all_ind["Indicador"] == label]
    if df_ind.empty:
        print(f"Indicador '{label}': sem dados. Pulando.")
        continue

    pasta_indicador = os.path.join(OUT_DIR_ROOT, slug)
    os.makedirs(pasta_indicador, exist_ok=True)
    print(f"\n{'='*15} Indicador: {label} (pasta: {slug}/) {'='*15}")

    for paradigma in ORDEM_PARADIGMAS:
        df_p = df_ind[df_ind["Paradigma"] == paradigma]
        if df_p.empty:
            continue

        df_proposta = df_p[df_p["Grupo"] == "Proposta"]
        df_resto = df_p[df_p["Grupo"].isin(["Literatura", "Descritiva"])]

        if df_proposta.empty or df_resto.empty:
            continue

        nome_a = f"{paradigma}_Proposta"
        nome_b = f"{paradigma}_Resto"

        caminho_a = os.path.join(pasta_indicador, f"DuplaFronteira_{nome_a}_{timestamp}.png")
        pareto_a = plot_fronteira(
            df_proposta,
            f"{label} -- DuplaFronteira_{nome_a}\nFronteira da Proposta (CEQ) -- Kneedle (círculo) e Estatístico (quadrado)",
            caminho_a,
        )

        caminho_b = os.path.join(pasta_indicador, f"DuplaFronteira_{nome_b}_{timestamp}.png")
        pareto_b = plot_fronteira(
            df_resto,
            f"{label} -- DuplaFronteira_{nome_b}\nFronteira de Literatura + Descritivas -- Kneedle (círculo) e Estatístico (quadrado)",
            caminho_b,
            cores_por_subgrupo={"Literatura": COR_GRUPO["Literatura"], "Descritiva": COR_GRUPO["Descritiva"]},
        )

        tab_a = tabela_pontos_na_fronteira(df_proposta)
        tab_b = tabela_pontos_na_fronteira(df_resto)
        if not tab_a.empty:
            tab_a.to_excel(os.path.join(pasta_indicador, f"DuplaFronteira_Tabela_{nome_a}_{timestamp}.xlsx"), index=False)
        if not tab_b.empty:
            tab_b.to_excel(os.path.join(pasta_indicador, f"DuplaFronteira_Tabela_{nome_b}_{timestamp}.xlsx"), index=False)

        exportar_todos_os_pontos(df_proposta, os.path.join(pasta_indicador, f"DuplaFronteira_TodosPontos_{nome_a}_{timestamp}.xlsx"))
        exportar_todos_os_pontos(df_resto, os.path.join(pasta_indicador, f"DuplaFronteira_TodosPontos_{nome_b}_{timestamp}.xlsx"))

        valores_a = pareto_a["Valor"].values
        valores_b = pareto_b["Valor"].values
        if len(valores_a) >= 1 and len(valores_b) >= 1 and not (len(valores_a) == 1 and len(valores_b) == 1):
            try:
                stat, p = mannwhitneyu(valores_a, valores_b, alternative="two-sided")
                sig = bool(p <= 0.05)
            except ValueError:
                p, sig = np.nan, False
        else:
            p, sig = np.nan, False

        media_a = float(np.mean(valores_a)) if len(valores_a) else np.nan
        media_b = float(np.mean(valores_b)) if len(valores_b) else np.nan
        quem_venceu = (
            "Proposta" if (not pd.isna(media_a) and not pd.isna(media_b) and media_a > media_b)
            else "Resto" if not pd.isna(media_b) else "--"
        )

        resumo_todos_indicadores.append({
            "Indicador": label,
            "Paradigma": paradigma,
            "N Pontos Fronteira Proposta": len(valores_a),
            "N Pontos Fronteira Resto": len(valores_b),
            "Média Proposta": round(media_a, 4) if not pd.isna(media_a) else np.nan,
            "Média Resto": round(media_b, 4) if not pd.isna(media_b) else np.nan,
            "p-valor (Mann-Whitney)": round(float(p), 4) if not pd.isna(p) else np.nan,
            "Diferença Significativa (p<=0.05)": sig,
            "Quem Venceu": quem_venceu,
        })

    # Comparação deste indicador, todos os paradigmas, salva dentro da própria subpasta
    df_comp_indicador = pd.DataFrame([r for r in resumo_todos_indicadores if r["Indicador"] == label])
    if not df_comp_indicador.empty:
        caminho_comp = os.path.join(pasta_indicador, f"DuplaFronteira_Comparacao_PropostaVsResto_{timestamp}.xlsx")
        df_comp_indicador.drop(columns="Indicador").to_excel(caminho_comp, index=False)
        print(df_comp_indicador.drop(columns="Indicador").to_string(index=False))

print(f"\nProcessamento concluído para {len([l for l, s in INDICADORES if not df_all_ind[df_all_ind['Indicador']==l].empty])} indicador(es).")



=============== Indicador: Acurácia Balanceada (pasta: Acuracia_Balanceada/) ===============
   Paradigma  N Pontos Fronteira Proposta  N Pontos Fronteira Resto  Média Proposta  Média Resto  p-valor (Mann-Whitney)  Diferença Significativa (p<=0.05) Quem Venceu
          DT                            7                        10          0.5771       0.5924                  0.4747                              False       Resto
          RF                            3                         4          0.5140       0.5203                  0.6286                              False       Resto
          GB                            4                         7          0.5302       0.5480                  0.3152                              False       Resto
         SVM                            1                         1          0.5000       0.5000                     NaN                              False       Resto
         MLP                            6                         

### 4. Resumo consolidado -- todos os indicadores × todos os paradigmas

Esta é a tabela que responde diretamente à pergunta "onde e em quais casos minhas
métricas têm vantagem/desvantagem": uma linha por combinação (Indicador, Paradigma),
com quem venceu e se a diferença é estatisticamente significativa. A segunda aba
(`Resumo_Vencedor`) é uma tabela pivotada (Indicador × Paradigma) só com o resultado
final, para uma leitura rápida de onde a Proposta vence/perde/empata -- um `*` ao lado
marca as vitórias/derrotas estatisticamente significativas (p ≤ 0,05); sem `*`, a
diferença pode ser só ruído amostral.

In [4]:
df_resumo_geral = pd.DataFrame(resumo_todos_indicadores)

if df_resumo_geral.empty:
    print("Nenhuma comparação pôde ser calculada -- confira se há dados suficientes "
          "de Proposta e Resto para algum indicador/paradigma.")
else:
    caminho_resumo = os.path.join(OUT_DIR_ROOT, f"Resumo_Todos_Indicadores_{timestamp}.xlsx")

    # --- Aba 2: tabela pivotada Indicador x Paradigma, só com o resultado final ---
    def rotulo_resultado(row):
        marca = "*" if row["Diferença Significativa (p<=0.05)"] else ""
        if row["Quem Venceu"] == "--":
            return "--"
        return f"{row['Quem Venceu']}{marca}"

    df_resumo_geral["Resultado"] = df_resumo_geral.apply(rotulo_resultado, axis=1)
    df_pivot = df_resumo_geral.pivot(index="Indicador", columns="Paradigma", values="Resultado")
    ordem_colunas_pivot = [p for p in ORDEM_PARADIGMAS if p in df_pivot.columns]
    df_pivot = df_pivot[ordem_colunas_pivot]
    ordem_indicadores_pivot = [label for label, _ in INDICADORES if label in df_pivot.index]
    df_pivot = df_pivot.loc[ordem_indicadores_pivot]

    with pd.ExcelWriter(caminho_resumo, engine="openpyxl") as writer:
        df_resumo_geral.drop(columns="Resultado").to_excel(writer, sheet_name="Detalhado", index=False)
        df_pivot.to_excel(writer, sheet_name="Resumo_Vencedor")

    print(f"Resumo consolidado salvo em: {caminho_resumo}")
    print("   (aba 'Detalhado': 1 linha por Indicador x Paradigma | "
          "aba 'Resumo_Vencedor': tabela pivotada, 'Proposta*'/'Resto*' = diferença significativa)\n")

    n_proposta_venceu = (df_resumo_geral["Quem Venceu"] == "Proposta").sum()
    n_resto_venceu = (df_resumo_geral["Quem Venceu"] == "Resto").sum()
    n_sig = df_resumo_geral["Diferença Significativa (p<=0.05)"].sum()
    print(f"Em {len(df_resumo_geral)} comparações (indicador x paradigma): "
          f"Proposta venceu em {n_proposta_venceu}, Resto venceu em {n_resto_venceu} "
          f"({n_sig} com diferença estatisticamente significativa).\n")

    print(df_pivot.to_string())


✅ Resumo consolidado salvo em: c:\Users\USUARIO\Desktop\doutorado\Códigos para o Artigo\Resultados_DuplaFronteira_PorIndicador\Resumo_Todos_Indicadores_20261023_125204.xlsx
   (aba 'Detalhado': 1 linha por Indicador x Paradigma | aba 'Resumo_Vencedor': tabela pivotada, 'Proposta*'/'Resto*' = diferença significativa)

📊 Em 77 comparações (indicador x paradigma): Proposta venceu em 14, Resto venceu em 63 (0 com diferença estatisticamente significativa).

Paradigma                      DT        RF        GB    SVM    MLP       GNB LogReg_Lasso
Indicador                                                                                 
Acurácia Balanceada         Resto     Resto     Resto  Resto  Resto  Proposta        Resto
Sensibilidade               Resto     Resto     Resto  Resto  Resto  Proposta        Resto
Especificidade           Proposta  Proposta  Proposta  Resto  Resto     Resto        Resto
Acurácia                 Proposta  Proposta  Proposta  Resto  Resto     Resto        Res